# Bab 9: k-Nearest Neighbor

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab09/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab09")
print("Folder kerja:", os.getcwd())

## Listing 9.1: $k$-NN yang ditulis sendiri

`kode/bab09/knn_numpy.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

DATA = "kode/data/"
biji = pd.read_csv(DATA + "mutu_biji.csv", parse_dates=["tanggal"])
rasa = ["aroma", "flavor", "aftertaste", "acidity", "body", "balance",
        "overall"]
latih = biji[biji["tanggal"] < "2025-01-01"]
uji = biji[biji["tanggal"] >= "2025-01-01"]
Xl, yl = latih[rasa].to_numpy(), latih["asal"].to_numpy()
Xu, yu = uji[rasa].to_numpy(), uji["asal"].to_numpy()


def knn(Xl, yl, Xu, k):
    d = ((Xu[:, None, :] - Xl[None, :, :]) ** 2).sum(-1)
    tetangga = np.argsort(d, axis=1)[:, :k]
    pred = []
    for baris in yl[tetangga]:
        nilai, cacah = np.unique(baris, return_counts=True)
        pred.append(nilai[np.argmax(cacah)])
    return np.array(pred)


for k in [1, 5, 25]:
    sendiri = (knn(Xl, yl, Xu, k) == yu).mean()
    sk = KNeighborsClassifier(k).fit(Xl, yl).score(Xu, yu)
    print(f"k={k:2d}: NumPy {sendiri:.3f}  scikit-learn {sk:.3f}")

## Listing 9.2: Pengaruh skala dan atribut tak relevan

`kode/bab09/skala.py`

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

DATA = "kode/data/"
biji = pd.read_csv(DATA + "mutu_biji.csv")
rasa = ["aroma", "flavor", "aftertaste", "acidity", "body", "balance",
        "overall"]
fisik = ["kadar_air", "cacat_primer", "cacat_sekunder", "ukuran_screen",
         "densitas_gl"]
print(biji[fisik + ["acidity"]].agg(["min", "max", "std"]).round(2)
      .to_string())
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print("\natribut          tanpa skala  min-maks  skor-z")
for nama, kol in [("rasa (7)", rasa), ("fisik (5)", fisik),
                  ("rasa + fisik", rasa + fisik)]:
    hasil = []
    for skala in [None, MinMaxScaler(), StandardScaler()]:
        m = KNeighborsClassifier(25)
        if skala is not None:
            m = make_pipeline(skala, m)
        hasil.append(cross_val_score(m, biji[kol], biji["asal"],
                                     cv=cv).mean())
    print(f"{nama:15s}  {hasil[0]:.3f}       {hasil[1]:.3f}"
          f"     {hasil[2]:.3f}")

## Listing 9.3: Akurasi menurut $k$ dan pembobotan

`kode/bab09/pilih_k.py`

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA = "kode/data/"
biji = pd.read_csv(DATA + "mutu_biji.csv")
rasa = ["aroma", "flavor", "aftertaste", "acidity", "body", "balance",
        "overall"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
baris = []
for k in [1, 3, 5, 9, 15, 25, 35, 51, 75, 101, 151, 201, 301]:
    r = {"k": k}
    for w in ["uniform", "distance"]:
        m = make_pipeline(StandardScaler(),
                          KNeighborsClassifier(k, weights=w))
        s = cross_validate(m, biji[rasa], biji["asal"], cv=cv,
                           return_train_score=True)
        r[w] = s["test_score"].mean()
        if w == "uniform":
            r["latih"] = s["train_score"].mean()
    baris.append(r)
tab = pd.DataFrame(baris)
print(tab.round(3).to_string(index=False))
tab.to_csv("gambar/bab09/pilih_k.dat", sep=" ", index=False)

## Listing 9.4: Kutukan dimensi pada jarak

`kode/bab09/kutukan.py`

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
baris = []
for p in [1, 2, 5, 10, 20, 50, 100, 200, 500]:
    X = rng.uniform(size=(1000, p))
    q = rng.uniform(size=(50, p))
    d = np.sqrt(((q[:, None, :] - X[None, :, :]) ** 2).sum(-1))
    rasio = (d.min(1) / d.max(1)).mean()
    baris.append((p, rasio))
    print(f"p={p:3d}: jarak terdekat / terjauh = {rasio:.3f}")
pd.DataFrame(baris, columns=["p", "rasio"]).to_csv(
    "gambar/bab09/kutukan.dat", sep=" ", index=False)

## Listing 9.5: $k$-NN regresi untuk skor cupping

`kode/bab09/knn_regresi.py`

In [ ]:
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA = "kode/data/"
biji = pd.read_csv(DATA + "mutu_biji.csv")
fisik = ["kadar_air", "cacat_primer", "cacat_sekunder", "ukuran_screen",
         "densitas_gl", "ketinggian_mdpl", "lama_simpan_hari"]
X, y = biji[fisik], biji["skor_total"]
cv = KFold(5, shuffle=True, random_state=0)
model = {"rata-rata": DummyRegressor(),
         "linier": make_pipeline(StandardScaler(), LinearRegression())}
for k in [1, 5, 10, 20, 40]:
    model[f"kNN k={k}"] = make_pipeline(StandardScaler(),
                                        KNeighborsRegressor(k))
model["kNN k=20 jarak"] = make_pipeline(
    StandardScaler(), KNeighborsRegressor(20, weights="distance"))
for nama, m in model.items():
    mae = -cross_val_score(m, X, y, cv=cv,
                           scoring="neg_mean_absolute_error").mean()
    print(f"{nama:15s} MAE {mae:.2f} poin")

## Listing 9.6: Supervised clustering dengan $k$-means per kelas

`kode/bab09/klaster_terawasi.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


class KlasterTerawasi(ClassifierMixin, BaseEstimator):
    def __init__(self, m=3):
        self.m = m

    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        self.classes_ = np.unique(y)
        pusat, label = [], []
        for c in self.classes_:
            km = KMeans(self.m, n_init=10, random_state=0).fit(X[y == c])
            pusat.append(km.cluster_centers_)
            label += [c] * self.m
        self.pusat_, self.label_ = np.vstack(pusat), np.array(label)
        return self

    def predict(self, X):
        d = ((np.asarray(X)[:, None, :] - self.pusat_) ** 2).sum(-1)
        return self.label_[d.argmin(1)]


DATA = "kode/data/"
biji = pd.read_csv(DATA + "mutu_biji.csv")
rasa = ["aroma", "flavor", "aftertaste", "acidity", "body", "balance",
        "overall"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for m in [1, 2, 4, 8, 16]:
    s = cross_val_score(make_pipeline(StandardScaler(),
                                      KlasterTerawasi(m)),
                        biji[rasa], biji["asal"], cv=cv).mean()
    print(f"{m:2d} klaster per kelas ({4 * m:3d} prototipe): {s:.3f}")
s = cross_val_score(make_pipeline(StandardScaler(),
                                  KNeighborsClassifier(25)),
                    biji[rasa], biji["asal"], cv=cv).mean()
print(f"kNN k=25 (1200 objek latih)       : {s:.3f}")

## Listing 9.7: Studi kasus asal biji dengan $k$-NN

`kode/bab09/studi_kasus.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA = "kode/data/"
biji = pd.read_csv(DATA + "mutu_biji.csv", parse_dates=["tanggal"])
rasa = ["aroma", "flavor", "aftertaste", "acidity", "body", "balance",
        "overall"]
latih = biji[biji["tanggal"] < "2025-01-01"]
uji = biji[biji["tanggal"] >= "2025-01-01"]
pipa = make_pipeline(StandardScaler(), KNeighborsClassifier())
kisi = {"kneighborsclassifier__n_neighbors": [5, 15, 25, 35, 51, 75],
        "kneighborsclassifier__weights": ["uniform", "distance"],
        "kneighborsclassifier__p": [1, 2]}
cari = GridSearchCV(pipa, kisi, cv=StratifiedKFold(
    5, shuffle=True, random_state=0)).fit(latih[rasa], latih["asal"])
print("terbaik:", {k.split("__")[1]: v
                   for k, v in cari.best_params_.items()},
      f"CV {cari.best_score_:.3f}")
pred = cari.predict(uji[rasa])
print(f"akurasi uji 2025: {(pred == uji['asal']).mean():.3f}\n")
urutan = ["Gayo", "Kerinci", "Toraja", "Flores"]
print(pd.DataFrame(confusion_matrix(uji["asal"], pred, labels=urutan),
                   index=urutan, columns=urutan).to_string())
print(classification_report(uji["asal"], pred, digits=3))

p = cari.predict_proba(uji[rasa]).max(1)
print("ambang suara  cakupan  akurasi")
for t in [0.0, 0.5, 0.6, 0.7, 0.8]:
    pilih = p >= t
    print(f"    {t:.1f}       {pilih.mean():.3f}    "
          f"{(pred[pilih] == uji['asal'].to_numpy()[pilih]).mean():.3f}")

model = cari.best_estimator_
satu = uji[uji["asal"] == "Kerinci"].iloc[[3]]
jarak, idx = model[-1].kneighbors(model[0].transform(satu[rasa]), 5)
tetangga = latih.iloc[idx[0]][["id_kiriman", "asal", "acidity", "body"]]
print(f"\nkiriman {satu['id_kiriman'].item()} (sebenarnya Kerinci),"
      f" lima tetangga terdekat:")
print(tetangga.assign(jarak=jarak[0].round(2)).to_string(index=False))